# Bài 07: Bài tập - Calibration & Cost-Sensitive Learning

**Mục tiêu:** Củng cố kiến thức về ý nghĩa của xác suất dự đoán, biểu đồ hiệu chỉnh và cách áp dụng `class_weight` để xử lý dữ liệu mất cân bằng.

---

### Phần 1: Câu hỏi lý thuyết

**Câu 1:** Một mô hình được gọi là "hiệu chỉnh tốt" (well-calibrated) nghĩa là gì? Tại sao việc có xác suất dự đoán đáng tin cậy lại quan trọng trong các ứng dụng thực tế?

**Câu 2:** Trên một biểu đồ hiệu chỉnh (Calibration Plot), nếu đường cong của mô hình nằm hoàn toàn **bên dưới** đường chéo hoàn hảo, điều đó cho thấy mô hình đang "quá tự tin" (over-confident) hay "thiếu tự tin" (under-confident)? Giải thích.

**Câu 3:** So sánh sự khác biệt trong triết lý tiếp cận giữa `SMOTE` và `class_weight='balanced'`. Một cái tác động lên dữ liệu, một cái tác động lên thuật toán. Hãy làm rõ sự khác biệt này.

**Câu 4:** Bạn đang sử dụng một thuật toán không hỗ trợ tham số `class_weight` (ví dụ: một phiên bản cũ của một thư viện nào đó). Bạn sẽ chọn phương pháp nào giữa `SMOTE` và `Cost-Sensitive Learning` để xử lý dữ liệu mất cân bằng? Tại sao?

**Câu 5:** Việc sử dụng `class_weight='balanced'` có ảnh hưởng đến thời gian huấn luyện mô hình không? So sánh với việc sử dụng `SMOTE`.

---

### Phần 2: Bài tập thực hành

**Bài tập 6: Áp dụng `class_weight` cho `RandomForestClassifier`**

Hoàn thành đoạn code dưới đây để huấn luyện một mô hình `RandomForestClassifier` sử dụng kỹ thuật Cost-Sensitive Learning và so sánh kết quả với mô hình gốc.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, balanced_accuracy_score

# 1. Tạo dữ liệu
X, y = make_classification(n_samples=5000, weights=[0.95, 0.05], n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 2. Huấn luyện mô hình RandomForest gốc
rf_base = RandomForestClassifier(random_state=42)
rf_base.fit(X_train, y_train)
y_pred_base = rf_base.predict(X_test)

print("--- Kết quả mô hình RandomForest GỐC ---")
print(classification_report(y_test, y_pred_base))
print(f"Balanced Accuracy (Gốc): {balanced_accuracy_score(y_test, y_pred_base):.4f}\n")

# 3. Huấn luyện mô hình RandomForest với class_weight='balanced'
# Điền vào chỗ trống
rf_weighted = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_weighted.fit(X_train, y_train)
y_pred_weighted = rf_weighted.predict(X_test)

print("--- Kết quả mô hình RandomForest với class_weight='balanced' ---")
print(classification_report(y_test, y_pred_weighted))
print(f"Balanced Accuracy (Weighted): {balanced_accuracy_score(y_test, y_pred_weighted):.4f}")

**Bài tập 7: Tự định nghĩa trọng số**

Dựa trên kết quả của `rf_weighted` ở bài 6, giả sử bạn muốn tăng `Recall` của lớp 1 lên cao hơn nữa, chấp nhận hy sinh một chút `Precision`. Bạn sẽ điều chỉnh dictionary `weights` như thế nào? Hãy thử nghiệm với một bộ trọng số bạn cho là hợp lý.

In [ ]:
# Gợi ý: Tăng trọng số của lớp 1 và giữ nguyên hoặc giảm nhẹ trọng số lớp 0
custom_weights = {0: 1, 1: 15} # Ví dụ: phạt lớp 1 nặng gấp 15 lần

rf_custom = RandomForestClassifier(random_state=42, class_weight=custom_weights)
rf_custom.fit(X_train, y_train)
y_pred_custom = rf_custom.predict(X_test)

print(f"--- Kết quả với class_weight tùy chỉnh {custom_weights} ---")
print(classification_report(y_test, y_pred_custom))
print(f"Balanced Accuracy (Custom): {balanced_accuracy_score(y_test, y_pred_custom):.4f}")

**Bài tập 8 (Tư duy):** So sánh kết quả của 3 mô hình Random Forest đã chạy. Bạn nhận thấy xu hướng gì khi tăng trọng số của lớp 1? Sự đánh đổi (trade-off) nào đang diễn ra?

---

### Phần 3: Đáp án và Giải thích

**Câu 1:**
Một mô hình được gọi là "hiệu chỉnh tốt" có nghĩa là xác suất mà nó dự đoán ra phản ánh đúng tần suất thực tế. Ví dụ, nếu chúng ta thu thập tất cả các dự đoán mà mô hình gán xác suất 70%, thì khoảng 70% trong số đó thực sự thuộc về lớp dương tính.
Điều này quan trọng vì trong thực tế, chúng ta thường không chỉ dựa vào nhãn 0/1 mà còn dựa vào mức độ chắc chắn của mô hình. Ví dụ:
- **Y tế:** Một bệnh nhân có 95% xác suất bị bệnh cần được can thiệp ngay lập tức, trong khi một bệnh nhân có 55% xác suất có thể cần làm thêm xét nghiệm.
- **Kinh doanh:** Một khách hàng có 80% khả năng mua hàng sẽ được gửi một mã giảm giá khác với một khách hàng chỉ có 30% khả năng.

**Câu 2:**
Nếu đường cong của mô hình nằm **bên dưới** đường chéo, điều đó có nghĩa là mô hình đang **quá tự tin (over-confident)**. Ví dụ, tại điểm mà mô hình dự đoán xác suất trung bình là 0.8 (trục x), tỷ lệ thực tế của lớp dương tính chỉ là 0.6 (trục y). Mô hình đã "thổi phồng" sự chắc chắn của mình.

**Câu 3:**
- **SMOTE (Tiếp cận dựa trên dữ liệu):** Triết lý của SMOTE là "Nếu dữ liệu huấn luyện bị sai, hãy sửa dữ liệu". Nó can thiệp trực tiếp vào tập huấn luyện bằng cách tạo ra các mẫu tổng hợp cho lớp thiểu số để làm cho tập dữ liệu trở nên cân bằng. Mô hình học máy sau đó sẽ học trên tập dữ liệu đã được "sửa đổi" này.
- **`class_weight` (Tiếp cận dựa trên thuật toán):** Triết lý của `class_weight` là "Dữ liệu cứ để nguyên, hãy sửa cách học của thuật toán". Nó không thay đổi dữ liệu. Thay vào đó, nó điều chỉnh hàm mất mát (loss function) của thuật toán, khiến cho việc phân loại sai một mẫu thuộc lớp thiểu số bị "phạt" nặng hơn. Thuật toán trở nên "nhạy cảm" hơn với các lỗi trên lớp thiểu số và tự điều chỉnh để tránh chúng.

**Câu 4:**
Nếu thuật toán không hỗ trợ `class_weight`, bạn chắc chắn phải chọn **SMOTE** (hoặc một kỹ thuật resampling khác). Lý do là vì `Cost-Sensitive Learning` là một tính năng nội tại của thuật toán, nếu nó không được hỗ trợ thì bạn không thể sử dụng. Ngược lại, `SMOTE` là một bước tiền xử lý dữ liệu, nó hoàn toàn độc lập với thuật toán bạn sẽ sử dụng sau đó. Bạn có thể dùng SMOTE để cân bằng dữ liệu rồi đưa vào bất kỳ mô hình phân loại nào.

**Câu 5:**
- **`class_weight`:** Hầu như không ảnh hưởng hoặc ảnh hưởng rất ít đến thời gian huấn luyện. Nó chỉ thay đổi cách tính toán lỗi trong mỗi bước lặp, không làm thay đổi kích thước của tập dữ liệu.
- **`SMOTE`:** Sẽ làm **tăng** thời gian huấn luyện. Lý do là vì SMOTE làm tăng đáng kể số lượng mẫu trong tập huấn luyện (tăng số mẫu lớp thiểu số cho bằng lớp đa số). Một tập huấn luyện lớn hơn đòi hỏi nhiều thời gian tính toán hơn để xử lý.
Do đó, `class_weight` nhanh hơn đáng kể so với `SMOTE`.

**Bài tập 6 (Đáp án):**
Phần cần điền là khởi tạo và huấn luyện mô hình `RandomForestClassifier` với tham số `class_weight='balanced'`.
```python
rf_weighted = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_weighted.fit(X_train, y_train)
y_pred_weighted = rf_weighted.predict(X_test)
```
Kết quả cho thấy `Recall` của lớp 1 và `Balanced Accuracy` tăng lên đáng kể so với mô hình gốc.

**Bài tập 7 (Đáp án):**
Để tăng `Recall` của lớp 1, chúng ta cần tăng trọng số phạt cho lớp 1. Ví dụ, `custom_weights = {0: 1, 1: 15}` hoặc `{0: 1, 1: 20}`. Việc chọn con số chính xác (`15`, `20`, hay `25`) là một quá trình tinh chỉnh (tuning) để tìm ra sự cân bằng tốt nhất cho bài toán cụ thể.

**Bài tập 8 (Đáp án):**
Khi tăng trọng số của lớp 1 (từ không có, đến `'balanced'`, đến ` {0: 1, 1: 15}`), ta sẽ thấy xu hướng sau:
- **`Recall` của lớp 1 tăng lên:** Mô hình ngày càng nỗ lực hơn để tìm ra các mẫu lớp 1.
- **`Precision` của lớp 1 giảm xuống:** Để tìm được nhiều mẫu lớp 1 hơn, mô hình trở nên "liều lĩnh" hơn và phân loại nhầm nhiều mẫu lớp 0 thành lớp 1 hơn (tăng FP).
- **`Recall` của lớp 0 giảm xuống:** Vì nhiều mẫu lớp 0 bị phân loại nhầm thành lớp 1.
Sự đánh đổi (trade-off) đang diễn ra chính là giữa **Recall của lớp 1** và **Precision của lớp 1** (cũng như Recall của lớp 0). Chúng ta đạt được khả năng phát hiện lớp thiểu số tốt hơn bằng cái giá của việc có nhiều báo động giả hơn.